## 1. 기본 패키지 불러오기
GNN graph process에 필요한 패키지를 불러옵니다. 기존 처리 로직과 경로는 그대로 유지했습니다.

In [ ]:
# GNN 산출물 생성에 필요한 패키지 불러오기
from pathlib import Path
import os
import gc
import warnings

import numpy as np
import pandas as pd
import polars as pl
import networkx as nx

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("networkx:", nx.__version__)

polars: 1.35.2
pandas: 2.2.2
numpy: 2.0.2
networkx: 3.6.1


## 2. 팀 공통 Google Drive 경로 설정
02/03번 산출물을 읽고, GNN용 CSV를 저장할 위치를 지정합니다. account_mapping은 새로 만들지 않고 기존 파일을 재사용합니다.

In [ ]:
from pathlib import Path

# ============================================================
# Google Drive 경로 설정
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
#
# Drive 폴더 구조:
# Graph_AML/
# └── data/
#     ├── raw/
#     └── processed/
#         ├── clean_base/
#         ├── ml_features/
#         └── gnn_inputs/
#
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# 모든 산출물은 Google Drive에 저장합니다.
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"

CLEAN_BASE_DIR = PROCESSED_DIR / "clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = PROCESSED_DIR / "gnn_inputs"
VALIDATION_DIR = GNN_DRIVE_DIR / "validation"


# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
CLEAN_BASE_DIR = CLEAN_BASE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    RAW_DIR,
    PROCESSED_DIR,
    CLEAN_BASE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
    VALIDATION_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("CLEAN_BASE_DIR :", CLEAN_BASE_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)
CONFIG = {
    "PROJECT_DIR": PROJECT_DIR,
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,

    # 02번 산출물 후보 경로. 새 Drive 구조를 우선 사용합니다.
    "CLEAN_BASE_CANDIDATES": [
        CLEAN_BASE_DIR / "clean_base.parquet",
        PROCESSED_DIR / "clean_base.parquet",
    ],
    "ACCOUNT_MAPPING_CANDIDATES": [
        CLEAN_BASE_DIR / "account_mapping.csv",
        PROCESSED_DIR / "account_mapping.csv",
    ],

    # 03번 산출물 후보 경로. split 컬럼 결합용
    "ML_EXP00_CANDIDATES": [
        ML_DIR / "ml_exp00.parquet",
        PROCESSED_DIR / "ml_exp00.parquet",
    ],
    "ML_EXP01_CANDIDATES": [
        ML_DIR / "ml_exp01.parquet",
        PROCESSED_DIR / "ml_exp01.parquet",
    ],

    # GNN 전달용 큰 산출물은 Drive에 저장
    "DRIVE_OUTPUT_DIR": GNN_DRIVE_DIR,

    "LOCAL_OUTPUT_DIR": GNN_DRIVE_DIR,

    # 기존 코드 호환용: GNN 주 산출물 위치는 Drive
    "OUTPUT_DIR": GNN_DRIVE_DIR,
    "VALIDATION_DIR": VALIDATION_DIR,

    # PageRank는 train split edge만 사용
    # 기본은 skip 금지
    "MAX_PAGERANK_EDGES": 5_000_000,
    # PageRank가 계산되지 않으면 기본적으로 실패 처리합니다.
    "ALLOW_PAGERANK_SKIP": False,
    "PAGERANK_ALPHA": 0.85,
    "PAGERANK_MAX_ITER": 100,
    "PAGERANK_TOL": 1e-06,

    # edge pair history는 edge-level 피처
    # 너무 크면 시간 부담으로 건너뜀
    "MAKE_SAFE_PAIR_HISTORY": True,
    "MAX_PAIR_HISTORY_ROWS": 1_000_000,
    "PAIR_HISTORY_WINDOWS": {
        "w1h": "1h",
        "w6h": "6h",
        "w1d": "1d",
        "w3d": "3d",
        "w7d": "7d",
    },

    "TIMESTAMP_COL": "timestamp",
    "TX_ID_COL": "tx_id",
    "LABEL_COL": "is_laundering",
    "AMOUNT_COL": "amount",
    "AMOUNT_RECEIVED_COL": "amount_received",
    "SENDER_COL": "sender_account_id",
    "RECEIVER_COL": "receiver_account_id",
    "FROM_BANK_COL": "sender_bank_id",
    "TO_BANK_COL": "receiver_bank_id",
    "RECEIVING_CURRENCY_COL": "receiving_currency",
}

CONFIG["DRIVE_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", CONFIG["PROJECT_DIR"])
print("DATA_DIR   :", CONFIG["DATA_DIR"])
print("DRIVE_OUT :", CONFIG["DRIVE_OUTPUT_DIR"])
print("input candidates:")
for k in ["CLEAN_BASE_CANDIDATES", "ACCOUNT_MAPPING_CANDIDATES", "ML_EXP00_CANDIDATES"]:
    print(k)
    for p in CONFIG[k]:
        print(" -", p, "exists=", Path(p).exists())

## 3. 보조 함수 준비
스키마 확인, 컬럼 목록 저장, 검증 리포트 작성에 쓰는 함수입니다. 산출물을 확인하기 쉽게 만드는 보조 코드입니다.

In [3]:
# ============================================================
# 작은 유틸
# ============================================================

MEMORY_RECORDS = []
GNN_CHECK_RECORDS = []


def get_memory_mb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


# 주요 단계별 메모리 사용량을 리포트용으로 기록합니다.
def record_memory(step, df=None):
    row_count = None
    col_count = None

    if df is not None:
        row_count = df.height if isinstance(df, pl.DataFrame) else len(df)
        col_count = len(df.columns)

    MEMORY_RECORDS.append({
        "step": step,
        "row_count": row_count,
        "col_count": col_count,
        "memory_mb": get_memory_mb(),
        "checked_at": pd.Timestamp.now().isoformat(),
    })
    print(f"[MEM] {step} | rows={row_count} | cols={col_count} | rss_mb={get_memory_mb():,.1f}")


def add_check(check_name, status, severity, details, n_failed_rows=0):
    GNN_CHECK_RECORDS.append({
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": str(details),
        "n_failed_rows": int(n_failed_rows or 0),
    })
    print(f"[{status}] {check_name} ({severity}) - {details}")


# P0 검증 실패 시 리포트에 남긴 뒤 즉시 중단합니다.
def fail_check(check_name, severity, details, n_failed_rows=0, raise_error=True):
    add_check(check_name, "FAIL", severity, details, n_failed_rows)
    if raise_error and severity == "P0":
        raise AssertionError(f"{check_name}: {details}")


# 후보 경로 중 실제 존재하는 파일을 선택합니다.
def resolve_existing_path(candidates, name):
    checked = []
    for path in candidates:
        path = Path(path)
        checked.append(str(path))
        if path.exists():
            print(f"{name}: {path}")
            return path
    raise FileNotFoundError(f"{name} 파일을 찾지 못했습니다. 확인한 경로: {checked}")


def normalize_name(x):
    return str(x).strip().lower().replace("_", " ").replace("-", " ")


def resolve_column(df, preferred, candidates=None, required=True):
    if preferred in df.columns:
        return preferred

    candidates = candidates or []
    norm_to_real = {normalize_name(c): c for c in df.columns}
    for cand in [preferred] + candidates:
        real = norm_to_real.get(normalize_name(cand))
        if real is not None:
            return real

    if required:
        raise KeyError(f"필수 컬럼을 찾지 못했습니다: {preferred} / candidates={candidates} / columns={df.columns}")
    return None


# 한글이 깨지지 않도록 utf-8-sig로 CSV를 저장합니다.
def write_csv(df, path):
    path = Path(path)
    if isinstance(df, pl.DataFrame):
        df.write_csv(path)
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")
    print("saved:", path)
    return path


def parse_window_to_ns(window_text):
    td = pd.Timedelta(window_text)
    return int(td.value)

## 4. 입력 파일 로드
GNN용 edge/node 파일을 만들기 위해 clean base, ML split 정보, account mapping을 읽습니다.

In [ ]:
# ============================================================
# 입력 파일 로드
# ============================================================

# 01번 clean_base 후보 경로를 02번과 같은 우선순위로 찾습니다.
clean_base_path = resolve_existing_path(CONFIG["CLEAN_BASE_CANDIDATES"], "clean_base")
# Step 1 account_mapping을 재사용합니다.
account_mapping_path = resolve_existing_path(CONFIG["ACCOUNT_MAPPING_CANDIDATES"], "account_mapping")
ml_exp00_path = resolve_existing_path(CONFIG["ML_EXP00_CANDIDATES"], "ml_exp00")

# GNN edge 생성의 기준이 되는 clean_base를 읽습니다.
clean_base = pl.read_parquet(clean_base_path)
account_mapping = pl.read_csv(account_mapping_path)
ml_exp00 = pl.read_parquet(ml_exp00_path)

record_memory("load_clean_base", clean_base)
record_memory("load_account_mapping", account_mapping)
record_memory("load_ml_exp00", ml_exp00)

print("clean_base shape:", clean_base.shape)
print("account_mapping shape:", account_mapping.shape)
print("ml_exp00 shape:", ml_exp00.shape)

clean_base.head(3)

clean_base: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/clean_base.parquet
account_mapping: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/account_mapping.csv
ml_exp00: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_exp00.parquet
[MEM] load_clean_base | rows=5078345 | cols=23 | rss_mb=9,871.0
[MEM] load_account_mapping | rows=515088 | cols=2 | rss_mb=9,859.0
[MEM] load_ml_exp00 | rows=5078345 | cols=18 | rss_mb=9,849.0
clean_base shape: (5078345, 23)
account_mapping shape: (515088, 2)
ml_exp00 shape: (5078345, 18)


tx_id,source_row_number,timestamp,timestamp_raw,sender_account_id,receiver_account_id,sender_bank_id,receiver_bank_id,amount,is_laundering,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,amount_received,amount_paid
u32,u32,datetime[μs],str,str,str,str,str,f64,i8,str,str,str,str,str,f64,str,f64,str,str,i64,f64,f64
0,2,2022-09-01 00:00:00,"""2022/09/01 00:00""","""03209|8000F4670""","""03209|8000F4670""","""03209""","""03209""",14675.57,0,"""2022/09/01 00:00""","""03209""","""8000F4670""","""03209""","""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0,14675.57,14675.57
1,17,2022-09-01 00:00:00,"""2022/09/01 00:00""","""01420|8005DFEB0""","""01420|8005DFEB0""","""01420""","""01420""",897.37,0,"""2022/09/01 00:00""","""01420""","""8005DFEB0""","""01420""","""8005DFEB0""",897.37,"""US Dollar""",897.37,"""US Dollar""","""Reinvestment""",0,897.37,897.37
2,158,2022-09-01 00:00:00,"""2022/09/01 00:00""","""010|8000F6850""","""010|8000F6850""","""010""","""010""",99986.94,0,"""2022/09/01 00:00""","""010""","""8000F6850""","""010""","""8000F6850""",99986.94,"""US Dollar""",99986.94,"""US Dollar""","""Reinvestment""",0,99986.94,99986.94


## 5. 기본 edge list 생성
거래 1건을 그래프의 edge로 보고, 송신 계좌에서 수신 계좌로 이어지는 기본 거래 edge를 만듭니다.

In [ ]:
# ============================================================
# 컬럼 정리
# ============================================================

COL = {
    "tx_id": resolve_column(clean_base, CONFIG["TX_ID_COL"], ["transaction_id", "id"]),
    "timestamp": resolve_column(clean_base, CONFIG["TIMESTAMP_COL"], ["Timestamp", "time"]),
    "label": resolve_column(clean_base, CONFIG["LABEL_COL"], ["label", "Is Laundering", "Is_Laundering"]),
    "amount": resolve_column(clean_base, CONFIG["AMOUNT_COL"], ["Amount Paid", "amount_paid"]),
    "sender": resolve_column(clean_base, CONFIG["SENDER_COL"], ["from_id", "sender", "from_account"]),
    "receiver": resolve_column(clean_base, CONFIG["RECEIVER_COL"], ["to_id", "receiver", "to_account"]),
    "from_bank": resolve_column(clean_base, CONFIG["FROM_BANK_COL"], ["From Bank", "from_bank"], required=False),
    "to_bank": resolve_column(clean_base, CONFIG["TO_BANK_COL"], ["To Bank", "to_bank"], required=False),
    "payment_format": resolve_column(clean_base, "payment_format", ["Payment Format", "payment format"], required=False),
    "currency": resolve_column(clean_base, "payment_currency", ["Payment Currency", "currency", "amount_currency"], required=False),
    "receiving_currency": resolve_column(clean_base, CONFIG["RECEIVING_CURRENCY_COL"], ["Received Currency", "received_currency", "receiving_currency"], required=False),
    "amount_received": resolve_column(clean_base, CONFIG["AMOUNT_RECEIVED_COL"], ["Amount Received", "amount_received", "amount_recv"], required=False),
}

MAP_COL = {
    "account_id": resolve_column(account_mapping, "account_id", ["account", "node_id"]),
    "node_idx": resolve_column(account_mapping, "node_idx", ["idx", "node_index"]),
}

SPLIT_TX_COL = resolve_column(ml_exp00, "tx_id", [COL["tx_id"]])
SPLIT_COL = resolve_column(ml_exp00, "split", ["data_split"])

print("COL:", COL)
print("MAP_COL:", MAP_COL)
print("split source columns:", SPLIT_TX_COL, SPLIT_COL)

## 6. formatted_transactions.csv 생성
GNN 담당자가 가장 먼저 확인할 수 있는 기본 edge list 파일입니다. 거래 방향과 라벨, split을 확인하는 용도입니다.

In [ ]:
# ============================================================
# split 결합 및 기본 edge list 생성
# ============================================================

# split은 02번 결과를 기준으로 사용한다.
split_df = (
    ml_exp00
    .select([
        pl.col(SPLIT_TX_COL).cast(pl.Utf8).alias("tx_id_join"),
        pl.col(SPLIT_COL).cast(pl.Utf8).alias("split"),
    ])
    .unique(subset=["tx_id_join"], keep="first")
)

base = (
    clean_base
    .with_columns([
        pl.col(COL["tx_id"]).cast(pl.Utf8).alias("tx_id"),
        pl.col(COL["timestamp"]).cast(pl.Datetime).alias("timestamp"),
        pl.col(COL["label"]).cast(pl.Int8).alias("label"),
        pl.col(COL["amount"]).cast(pl.Float64).alias("amount"),
        pl.col(COL["sender"]).cast(pl.Utf8).alias("from_id"),
        pl.col(COL["receiver"]).cast(pl.Utf8).alias("to_id"),
    ])
    .join(split_df, left_on="tx_id", right_on="tx_id_join", how="left")
)

missing_split = base.filter(pl.col("split").is_null()).height
if missing_split:
    fail_check("split_join_coverage", "P0", f"02번 ml_exp00 기준 split 매칭 실패 row={missing_split}", missing_split)
else:
    add_check("split_join_coverage", "PASS", "P0", "all tx_id matched to split from ml_exp00", 0)

mapping_std = account_mapping.select([
    pl.col(MAP_COL["account_id"]).cast(pl.Utf8).alias("account_id"),
    pl.col(MAP_COL["node_idx"]).cast(pl.Int64).alias("node_idx"),
])

base = (
    base
    .join(mapping_std.rename({"account_id": "from_id", "node_idx": "from_idx"}), on="from_id", how="left")
    .join(mapping_std.rename({"account_id": "to_id", "node_idx": "to_idx"}), on="to_id", how="left")
)

optional_cols = []
if COL["from_bank"]:
    optional_cols.append(pl.col(COL["from_bank"]).cast(pl.Utf8).alias("from_bank"))
if COL["to_bank"]:
    optional_cols.append(pl.col(COL["to_bank"]).cast(pl.Utf8).alias("to_bank"))
if COL["currency"]:
    optional_cols.append(pl.col(COL["currency"]).cast(pl.Utf8).alias("cat__payment_currency__code"))
if COL["receiving_currency"]:
    optional_cols.append(pl.col(COL["receiving_currency"]).cast(pl.Utf8).alias("cat__receiving_currency__code"))
if COL["payment_format"]:
    optional_cols.append(pl.col(COL["payment_format"]).cast(pl.Utf8).alias("cat__payment_format__code"))
if COL["amount_received"]:
    optional_cols.append(pl.col(COL["amount_received"]).cast(pl.Float64).alias("amount_received"))
else:
    optional_cols.append(pl.lit(None).cast(pl.Float64).alias("amount_received"))

base = base.with_columns(optional_cols) if optional_cols else base

# 기본 edge list를 from_idx, to_idx 기준으로 만듭니다.
formatted_transactions = (
    base
    .select(["tx_id", "from_id", "to_id", "from_idx", "to_idx", "amount", "timestamp", "split", "label"])
    .sort(["timestamp", "tx_id"])
)

record_memory("formatted_transactions", formatted_transactions)
formatted_transactions.head(5)

## 7. edge feature 정리
edge 단위로 의미 있는 거래 금액, 통화, 시간, split, label 관련 컬럼을 정리합니다. node feature와 섞지 않는 것이 핵심입니다.

In [ ]:
# ============================================================
# formatted_transactions_gf 기본 컬럼
# ============================================================

edge_cols = ["from_id", "to_id", "amount", "split", "label"]
for c in ["cat__payment_currency__code", "cat__receiving_currency__code", "cat__payment_format__code"]:
    if c in base.columns:
        edge_cols.append(c)

formatted_transactions_gf = (
    base
    .with_columns([
        (pl.col(COL["from_bank"]).cast(pl.Utf8) + "_" + pl.col(COL["sender"]).cast(pl.Utf8)).alias("from_id"),
        (pl.col(COL["to_bank"]).cast(pl.Utf8) + "_" + pl.col(COL["receiver"]).cast(pl.Utf8)).alias("to_id"),
        pl.when(pl.col("amount") > 0)
        .then(pl.col("amount").log1p())
        .otherwise(0.0)
        .alias("amount__current__log1p"),
        pl.col("amount_received").fill_null(0.0).alias("amount_received__current__value"),
        pl.when(
            pl.col("amount_received").is_not_null() & (pl.col("amount_received") > 0)
        )
        .then(pl.col("amount_received").log1p())
        .otherwise(0.0)
        .alias("amount_received__current__log1p"),
        pl.col("timestamp").dt.hour().cast(pl.Int32).alias("time__row__hour"),
        pl.col("timestamp").dt.weekday().cast(pl.Int32).alias("time__row__dayofweek"),
        (pl.col("timestamp").dt.weekday() >= 5).cast(pl.Int32).alias("time__row__is_weekend"),
    ])
    .sort(["timestamp", COL["tx_id"]])
    .select(
        edge_cols[:2]
        + ["amount__current__log1p", "amount_received__current__value", "amount_received__current__log1p",
           "time__row__hour", "time__row__dayofweek", "time__row__is_weekend"]
        + edge_cols[2:]
    )
)

record_memory("formatted_transactions_gf_base", formatted_transactions_gf)
formatted_transactions_gf.head(5)

## 8. formatted_transactions_gf.csv 생성
GNN 학습에 사용할 edge feature 파일입니다. PageRank, degree 같은 node feature는 여기에 반복해서 붙이지 않습니다.

In [ ]:
record_memory("formatted_transactions_gf", formatted_transactions_gf)
formatted_transactions_gf.head(5)

## 9. node feature 대상 계좌 정리
송신/수신에 등장한 모든 계좌를 node로 정리합니다. 계좌 단위 피처는 별도 node table에 저장합니다.

In [ ]:
# ============================================================
# account_node_features 생성
# node feature는 여기로만 모은다.
# PageRank/degree/account-level 통계는 edge 파일에 붙이지 않는다.
# ============================================================

train_edges = formatted_transactions.filter(pl.col("split") == "train")
train_edge_count = train_edges.height
all_node_count = mapping_std.height

print("train edges:", train_edge_count)
print("all nodes:", all_node_count)

# train 기준 in/out amount
out_stats = (
    train_edges
    .group_by("from_idx")
    .agg([
        pl.col("amount").sum().alias("train_only_out_amount_sum_raw"),
    ])
    .rename({"from_idx": "node_idx"})
)

in_stats = (
    train_edges
    .group_by("to_idx")
    .agg([
        pl.col("amount").sum().alias("train_only_in_amount_sum_raw"),
    ])
    .rename({"to_idx": "node_idx"})
)

node_features = (
    mapping_std
    .rename({"account_id": "account_id"})
    .join(out_stats, on="node_idx", how="left")
    .join(in_stats, on="node_idx", how="left")
    .with_columns([
        pl.col("train_only_out_amount_sum_raw").fill_null(0.0),
        pl.col("train_only_in_amount_sum_raw").fill_null(0.0),
    ])
    .with_columns([
        pl.col("train_only_out_amount_sum_raw").log1p().alias("train_only_out_amount_sum_log1p"),
        pl.col("train_only_in_amount_sum_raw").log1p().alias("train_only_in_amount_sum_log1p"),
    ])
    .drop(["train_only_out_amount_sum_raw", "train_only_in_amount_sum_raw"])
)

record_memory("node_features_basic_stats", node_features)
node_features.head(5)

## 10. train split 기반 PageRank/degree 계산
누수 방지를 위해 PageRank와 degree는 반드시 train split edge만 사용해 계산합니다. 전체 데이터 기준 centrality 계산은 금지입니다.

In [ ]:
# 이 셀은 train split edge만 사용해 node-level PageRank와 degree를 계산합니다.
# ============================================================
# train-only PageRank / degree
# ============================================================

pagerank_skipped_reason = ""

G_degree = nx.DiGraph()
G_degree.add_nodes_from(mapping_std["node_idx"].to_list())

train_pairs = train_edges.select(["from_idx", "to_idx"]).unique().to_pandas()
G_degree.add_edges_from(train_pairs[["from_idx", "to_idx"]].itertuples(index=False, name=None))

degree_pdf = pd.DataFrame({
    "node_idx": list(G_degree.nodes()),
    "train_only_in_degree": [G_degree.in_degree(n) for n in G_degree.nodes()],
    "train_only_out_degree": [G_degree.out_degree(n) for n in G_degree.nodes()],
})
degree_pl = pl.from_pandas(degree_pdf).with_columns(pl.col("node_idx").cast(pl.Int64))

if len(train_pairs) > CONFIG["MAX_PAGERANK_EDGES"]:
    pagerank_skipped_reason = (
        "too_many_edges_for_networkx: "
        f"graph_edges={len(train_pairs):,}, max={CONFIG['MAX_PAGERANK_EDGES']:,}"
    )

    if not CONFIG.get("ALLOW_PAGERANK_SKIP", False):
        add_check(
            "train_only_pagerank_compute",
            "FAIL",
            "P0",
            pagerank_skipped_reason + ", ALLOW_PAGERANK_SKIP=False",
            len(train_pairs),
        )
        write_csv(
            pd.DataFrame(GNN_CHECK_RECORDS),
            CONFIG["VALIDATION_DIR"] / "gnn_schema_check.csv",
        )
        raise AssertionError(
            "PageRank 계산 skip. "
            "MAX_PAGERANK_EDGES를 늘리거나 ALLOW_PAGERANK_SKIP=True로 명시 필요. "
            + pagerank_skipped_reason
        )

    pr_pdf = pd.DataFrame({
        "node_idx": mapping_std["node_idx"].to_list(),
        "train_only_pagerank": np.zeros(mapping_std.height, dtype=float),
    })
    add_check(
        "train_only_pagerank_compute",
        "PASS",
        "P1",
        pagerank_skipped_reason + ", explicitly allowed by ALLOW_PAGERANK_SKIP=True",
        0,
    )
else:
    G = nx.DiGraph()
    G.add_nodes_from(mapping_std["node_idx"].to_list())
    G.add_edges_from(train_pairs[["from_idx", "to_idx"]].itertuples(index=False, name=None))
    pr = nx.pagerank(
        G,
        alpha=CONFIG["PAGERANK_ALPHA"],
        max_iter=CONFIG["PAGERANK_MAX_ITER"],
        tol=CONFIG["PAGERANK_TOL"],
        weight=None,
    )
    pr_pdf = pd.DataFrame({
        "node_idx": list(pr.keys()),
        "train_only_pagerank": list(pr.values()),
    })
    add_check(
        "train_only_pagerank_compute",
        "PASS",
        "P0",
        f"unweighted PageRank computed with train edges only; graph_edges={len(train_pairs):,}",
        0,
    )

pr_pl = pl.from_pandas(pr_pdf).with_columns(pl.col("node_idx").cast(pl.Int64))

account_node_features = (
    node_features
    .join(degree_pl, on="node_idx", how="left")
    .join(pr_pl, on="node_idx", how="left")
    .with_columns([
        pl.col("train_only_in_degree").fill_null(0).cast(pl.Int64),
        pl.col("train_only_out_degree").fill_null(0).cast(pl.Int64),
        pl.col("train_only_pagerank").fill_null(0.0),
    ])
    .select([
        "node_idx", "account_id",
        "train_only_pagerank",
        "train_only_in_degree", "train_only_out_degree",
        "train_only_in_amount_sum_log1p", "train_only_out_amount_sum_log1p",
    ])
    .sort("node_idx")
)

record_memory("account_node_features", account_node_features)
account_node_features.head(5)

## 11. account_node_features.csv 생성
계좌별 PageRank, degree 등 node feature를 저장합니다. edge 파일에 반복 저장하지 않아 메모리와 중복을 줄입니다.

In [ ]:
# ============================================================
# feature column metadata
# ============================================================

def dtype_of(df, col):
    return str(df.schema.get(col))


def edge_feature_metadata(df):
    rows = []

    metadata_cols = {
        "from_id",
        "to_id",
        "split",
        "label",
    }

    for col in df.columns:
        is_metadata = col in metadata_cols
        used = not is_metadata
        role = "metadata" if is_metadata else "edge_feature"

        if is_metadata:
            source = "metadata_or_edge_endpoint"
            risk = "not_model_feature"
            null_policy = "must_not_null" if col in ["from_id", "to_id"] else "not_applicable"
            notes = "Do not use as input feature"
        elif col.startswith("pair_"):
            source = "safe_pair_history"
            risk = "low_if_past_timestamp_only"
            null_policy = "fill_0_when_no_past_pair_history"
            notes = "computed with past_timestamp < current_timestamp"
        elif col in ["time__row__hour", "time__row__dayofweek", "time__row__is_weekend"]:
            source = "timestamp_derived"
            risk = "low"
            null_policy = "not_null"
            notes = "derived from timestamp; raw timestamp dropped"
        elif col in ["amount", "amount__current__log1p",
                     "amount_received__current__value", "amount_received__current__log1p",
                     "cat__payment_currency__code", "cat__receiving_currency__code", "cat__payment_format__code"]:
            source = "current_transaction_row"
            risk = "low"
            null_policy = "as_is_or_current_row_transform"
            notes = "current transaction attribute"
        else:
            source = "edge_feature_candidate"
            risk = "review"
            null_policy = "review"
            notes = "review before modeling"

        rows.append({
            "column_name": col,
            "role": role,
            "used_as_edge_feature": bool(used),
            "dtype": dtype_of(df, col),
            "source": source,
            "leakage_risk": risk,
            "null_policy": null_policy,
            "notes": notes,
        })

    return pd.DataFrame(rows)


def node_feature_metadata(df):
    rows = []
    for col in df.columns:
        used = col not in ["node_idx", "account_id"]
        if col == "train_only_pagerank":
            source = "networkx_unweighted_pagerank"
            leakage_rule = "fit_on_train_split_edges_only"
            null_policy = "fill_0_for_val_test_only_node"
            notes = "weighted PageRank is TODO only"
        elif col in ["train_only_in_degree", "train_only_out_degree"]:
            source = "networkx_train_edge_graph"
            leakage_rule = "train_split_edges_only"
            null_policy = "fill_0_for_val_test_only_node"
            notes = "directed graph degree"
        elif col in ["train_only_in_amount_sum_log1p", "train_only_out_amount_sum_log1p"]:
            source = "polars_train_edge_aggregation"
            leakage_rule = "train_split_edges_only"
            null_policy = "fill_0_for_val_test_only_node"
            notes = "account-level node feature, not repeated in edge file"
        else:
            source = "identifier_or_status"
            leakage_rule = "not_model_feature" if not used else "review"
            null_policy = "not_applicable"
            notes = "kept for node lookup/status"

        rows.append({
            "column_name": col,
            "used_as_node_feature": bool(used),
            "dtype": dtype_of(df, col),
            "source": source,
            "leakage_rule": leakage_rule,
            "null_policy": null_policy,
            "notes": notes,
        })
    return pd.DataFrame(rows)


edge_feature_columns = edge_feature_metadata(formatted_transactions_gf)
node_feature_columns = node_feature_metadata(account_node_features)

edge_feature_columns.head(10)

## 12. edge/node feature column 목록 저장
GNN 담당자가 어떤 컬럼을 edge feature로 쓰고, 어떤 컬럼을 node feature로 써야 하는지 확인할 수 있도록 목록을 저장합니다.

In [ ]:
# ============================================================
# GNN 검증 함수
# ============================================================


def validate_gnn_mapping():
    dup_mapping = mapping_std.height - mapping_std.select(pl.col("account_id").n_unique()).item()
    dup_node = mapping_std.height - mapping_std.select(pl.col("node_idx").n_unique()).item()
    if dup_mapping or dup_node:
        fail_check("account_mapping_unique", "P0", f"account_dup={dup_mapping}, node_idx_dup={dup_node}", dup_mapping + dup_node)
    else:
        add_check("account_mapping_unique", "PASS", "P0", "account_id and node_idx are unique", 0)

    miss_from = formatted_transactions.filter(pl.col("from_idx").is_null()).height
    miss_to = formatted_transactions.filter(pl.col("to_idx").is_null()).height
    if miss_from or miss_to:
        fail_check("formatted_transactions_mapping_missing", "P0", f"from_missing={miss_from}, to_missing={miss_to}", miss_from + miss_to)
    else:
        add_check("formatted_transactions_mapping_missing", "PASS", "P0", "from_idx/to_idx are fully mapped", 0)


def validate_node_edge_feature_separation():
    forbidden_patterns = [
        "pagerank", "centrality", "degree",
        "train_only_in_amount_sum_log1p", "train_only_out_amount_sum_log1p",
    ]
    bad_cols = []
    for col in formatted_transactions_gf.columns:
        lower = col.lower()
        if any(p in lower for p in forbidden_patterns):
            bad_cols.append(col)

    if bad_cols:
        fail_check("node_edge_feature_separation", "P0", f"node feature columns found in edge file: {bad_cols}", len(bad_cols))
    else:
        add_check("node_edge_feature_separation", "PASS", "P0", "no PageRank/degree/account node features in formatted_transactions_gf", 0)


def validate_gnn_schema():
    tx_dup = formatted_transactions.height - formatted_transactions.select(pl.col("tx_id").n_unique()).item()
    if tx_dup:
        fail_check("tx_id_unique", "P0", f"duplicated tx_id rows={tx_dup}", tx_dup)
    else:
        add_check("tx_id_unique", "PASS", "P0", "tx_id is unique", 0)

    bad_label = formatted_transactions.filter(pl.col("label").is_null() | (~pl.col("label").is_in([0, 1]))).height
    if bad_label:
        fail_check("label_binary", "P0", f"bad label rows={bad_label}", bad_label)
    else:
        add_check("label_binary", "PASS", "P0", "label values are 0/1", 0)

    bad_split = formatted_transactions.filter(~pl.col("split").is_in(["train", "val", "test"])).height
    if bad_split:
        fail_check("split_values", "P0", f"bad split rows={bad_split}", bad_split)
    else:
        add_check("split_values", "PASS", "P0", "split values are train/val/test", 0)

    ts_order_check = (
        formatted_transactions
        .select([
            (
                pl.col("timestamp") < pl.col("timestamp").shift(1)
            ).fill_null(False).sum().alias("n_timestamp_order_violations")
        ])
    )

    n_ts_order_violations = int(ts_order_check["n_timestamp_order_violations"][0])

    if n_ts_order_violations > 0:
        fail_check(
            "timestamp_sorted",
            "P1",
            "formatted_transactions is not sorted by timestamp; violations={}".format(
                n_ts_order_violations
            ),
            n_ts_order_violations,
            raise_error=False,
        )
    else:
        add_check(
            "timestamp_sorted",
            "PASS",
            "P1",
            "formatted_transactions sorted by timestamp",
            0,
        )


def run_all_gnn_validations():
    validate_gnn_mapping()
    validate_node_edge_feature_separation()
    validate_gnn_schema()


run_all_gnn_validations()

## 13. GNN schema check 저장
필수 컬럼 존재 여부, row 수, null 여부 등을 점검한 결과를 저장합니다. 산출물 신뢰도를 확인하는 리포트입니다.

In [ ]:
# ============================================================
# 저장
# 검증 통과 후 저장
# ============================================================

DRIVE_OUT = CONFIG["DRIVE_OUTPUT_DIR"]
LOCAL_OUT = CONFIG["LOCAL_OUTPUT_DIR"]
VALIDATION_OUT = CONFIG["VALIDATION_DIR"]

write_csv(formatted_transactions, DRIVE_OUT / "formatted_transactions.csv")
write_csv(formatted_transactions_gf, DRIVE_OUT / "formatted_transactions_gf.csv")
write_csv(account_node_features, DRIVE_OUT / "account_node_features.csv")

# Step 1 account_mapping 그대로 복사. 새 매핑 생성 아님
write_csv(mapping_std.select(["account_id", "node_idx"]).sort("node_idx"), DRIVE_OUT / "account_mapping.csv")

write_csv(edge_feature_columns, DRIVE_OUT / "edge_feature_columns.csv")
write_csv(node_feature_columns, DRIVE_OUT / "node_feature_columns.csv")
write_csv(pd.DataFrame(GNN_CHECK_RECORDS), VALIDATION_OUT / "gnn_schema_check.csv")


print("저장 완료")

## 14. 최종 산출물 확인
생성된 GNN 파일 목록과 주요 요약을 출력합니다. 실행 후 이 셀을 보고 전달 가능한 상태인지 확인합니다.

In [ ]:
# ============================================================
# 마지막 확인
# ============================================================

expected_outputs = [
    (CONFIG["DRIVE_OUTPUT_DIR"], "formatted_transactions.csv"),
    (CONFIG["DRIVE_OUTPUT_DIR"], "formatted_transactions_gf.csv"),
    (CONFIG["DRIVE_OUTPUT_DIR"], "account_node_features.csv"),
    (CONFIG["DRIVE_OUTPUT_DIR"], "account_mapping.csv"),
    (CONFIG["DRIVE_OUTPUT_DIR"], "edge_feature_columns.csv"),
    (CONFIG["DRIVE_OUTPUT_DIR"], "node_feature_columns.csv"),
    (CONFIG["VALIDATION_DIR"],   "gnn_schema_check.csv"),
]

rows = []
for out_dir, fname in expected_outputs:
    path = Path(out_dir) / fname
    rows.append({
        "file_name": fname,
        "exists": path.exists(),
        "path": str(path),
        "size_mb": path.stat().st_size / 1024 / 1024 if path.exists() else None,
    })

output_check = pd.DataFrame(rows)
display(output_check)

print("\n실행 후 체크리스트")
print("1. gnn_schema_check.csv에서 from_idx/to_idx mapping PASS 확인")
print("2. account_node_features.csv row 수 == Step 1 account_mapping.csv row 수 확인")
print("3. formatted_transactions_gf.csv에 pagerank/degree/centrality가 없는지 확인")
print("4. PageRank는 train split edge만 사용했는지 확인")
print("5. edge_feature_columns.csv에서 role=metadata 컬럼은 input feature로 쓰지 않는지 확인")

print("\nTODO")
print("TODO: Stage 0 확장 후보 cur_vs_mean_ratio")
print("TODO: Pair 확장 후보 pair tgap, is_new_pair")

print("\n주요 shape")
print("formatted_transactions:", formatted_transactions.shape)
print("formatted_transactions_gf:", formatted_transactions_gf.shape)
print("account_node_features:", account_node_features.shape)
print("edge_feature_columns:", edge_feature_columns.shape)
print("node_feature_columns:", node_feature_columns.shape)

print("\nDrive 산출물 폴더:", CONFIG["DRIVE_OUTPUT_DIR"])